In [3]:
"""
FMP API를 사용하여 10년 이상 상장된 비금융, 비유틸리티 기업 필터링

필요한 FMP API 엔드포인트:
1. Stock List - 전체 상장 기업 목록
2. Company Profile - 개별 기업의 상장일(IPO date) 및 섹터 정보
"""

import requests
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import time

# ============================================================
# FMP API 설정
# ============================================================
FMP_API_KEY = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'
BASE_URL = "https://financialmodelingprep.com/api/v3"

# ============================================================
# 1. 전체 상장 기업 목록 가져오기
# ============================================================
def get_all_stock_list(api_key=FMP_API_KEY):
    """
    FMP에서 전체 상장 주식 목록을 가져옵니다.

    Returns:
        list: 티커 심볼 리스트
    """
    url = f"{BASE_URL}/stock/list?apikey={api_key}"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # 미국 주요 거래소만 필터링 (NYSE, NASDAQ)
        us_stocks = [
            stock['symbol']
            for stock in data
            if stock.get('exchangeShortName') in ['NYSE', 'NASDAQ', 'AMEX']
            and stock.get('type') == 'stock'  # ETF 제외
        ]

        print(f"총 {len(us_stocks)}개의 미국 상장 주식을 찾았습니다.")
        return us_stocks

    except Exception as e:
        print(f"주식 목록 가져오기 실패: {e}")
        return []


# ============================================================
# 2. 기업 프로필 정보 가져오기 (상장일, 섹터 등)
# ============================================================
def get_company_profile(ticker, api_key=FMP_API_KEY):
    """
    특정 티커의 기업 프로필을 가져옵니다.

    Args:
        ticker: 티커 심볼

    Returns:
        dict: 기업 정보 (ipoDate, sector, industry 등)
    """
    url = f"{BASE_URL}/profile/{ticker}?apikey={api_key}"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        if data and len(data) > 0:
            return data[0]
        return None

    except Exception as e:
        print(f"❌ {ticker} 프로필 가져오기 실패: {e}")
        return None


# ============================================================
# 3. 기업 필터링 함수
# ============================================================
def filter_mature_nonfinancial_companies(
    tickers,
    min_years=10,
    exclude_sectors=['Financials', 'Utilities', 'Financial Services'],
    api_key=FMP_API_KEY,
    batch_size=100,
    save_interval=500
):
    """
    10년 이상 상장된 비금융, 비유틸리티 기업 필터링

    Args:
        tickers: 티커 리스트
        min_years: 최소 상장 연수 (기본 10년)
        exclude_sectors: 제외할 섹터 리스트
        batch_size: 배치 크기
        save_interval: 중간 저장 간격

    Returns:
        pd.DataFrame: 필터링된 기업 정보
    """
    cutoff_date = datetime.now() - timedelta(days=min_years * 365)

    filtered_companies = []
    failed_tickers = []

    print(f"\n{'='*80}")
    print(f"필터링 조건:")
    print(f"  - 최소 상장일: {cutoff_date.strftime('%Y-%m-%d')} 이전")
    print(f"  - 제외 섹터: {', '.join(exclude_sectors)}")
    print(f"  - 총 티커 수: {len(tickers)}")
    print(f"{'='*80}\n")

    # 배치 처리
    for i in tqdm(range(0, len(tickers), batch_size), desc="기업 정보 수집 중"):
        batch_tickers = tickers[i:i + batch_size]

        for ticker in batch_tickers:
            try:
                profile = get_company_profile(ticker, api_key)

                if profile is None:
                    failed_tickers.append(ticker)
                    continue

                # 필수 정보 추출
                ipo_date_str = profile.get('ipoDate')
                sector = profile.get('sector', '')
                industry = profile.get('industry', '')

                # IPO 날짜 확인
                if not ipo_date_str:
                    continue

                ipo_date = datetime.strptime(ipo_date_str, '%Y-%m-%d')

                # 필터링 조건 확인
                is_old_enough = ipo_date <= cutoff_date
                is_not_excluded = sector not in exclude_sectors

                if is_old_enough and is_not_excluded:
                    filtered_companies.append({
                        'ticker': ticker,
                        'company_name': profile.get('companyName', ''),
                        'ipo_date': ipo_date_str,
                        'years_listed': (datetime.now() - ipo_date).days / 365.25,
                        'sector': sector,
                        'industry': industry,
                        'exchange': profile.get('exchangeShortName', ''),
                        'market_cap': profile.get('mktCap', None),
                        'country': profile.get('country', ''),
                    })

                # API 호출 제한 대응 (FMP는 분당 300 calls 제한)
                time.sleep(0.2)

            except Exception as e:
                failed_tickers.append(ticker)
                print(f"⚠️ {ticker} 처리 실패: {e}")
                continue

        # 중간 저장
        if (i + batch_size) % save_interval == 0 and filtered_companies:
            temp_df = pd.DataFrame(filtered_companies)
            temp_df.to_csv(f'filtered_companies_temp_{i}.csv', index=False)
            print(f"\n💾 중간 저장: {len(filtered_companies)}개 기업 저장됨")

    # 최종 결과
    if filtered_companies:
        df = pd.DataFrame(filtered_companies)
        df = df.sort_values('years_listed', ascending=False)

        print(f"\n{'='*80}")
        print(f"필터링 결과:")
        print(f"  - 조건 충족 기업: {len(df)}개")
        print(f"  - 실패한 티커: {len(failed_tickers)}개")
        print(f"{'='*80}\n")

        return df
    else:
        print("조건을 만족하는 기업이 없습니다.")
        return pd.DataFrame()


# ============================================================
# 4. 섹터별 통계 확인
# ============================================================
def show_sector_statistics(df):
    """필터링된 기업의 섹터별 통계 출력"""
    print("\n섹터별 기업 수:")
    print(df['sector'].value_counts())

    print("\n상장 연수 분포:")
    print(df['years_listed'].describe())

    print("\n거래소별 분포:")
    print(df['exchange'].value_counts())


# ============================================================
# 5. 데이터베이스 저장 (선택사항)
# ============================================================
def save_to_database(df, db_info, table_name='us_mature_companies'):
    """
    필터링된 기업 목록을 DB에 저장

    Args:
        df: 필터링된 기업 DataFrame
        db_info: DB 연결 정보 딕셔너리
        table_name: 테이블명
    """
    import pymysql

    try:
        conn = pymysql.connect(
            host=db_info['host'],
            user=db_info['user'],
            password=db_info['password'],
            database=db_info['database'],
            charset='utf8mb4'
        )
        cursor = conn.cursor()

        # 테이블 생성
        create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            ticker VARCHAR(20) PRIMARY KEY,
            company_name VARCHAR(255),
            ipo_date DATE,
            years_listed FLOAT,
            sector VARCHAR(100),
            industry VARCHAR(100),
            exchange VARCHAR(20),
            market_cap BIGINT,
            country VARCHAR(50),
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
        )
        """
        cursor.execute(create_table_sql)

        # 데이터 삽입
        insert_sql = f"""
        INSERT INTO {table_name}
        (ticker, company_name, ipo_date, years_listed, sector, industry, exchange, market_cap, country)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            company_name=VALUES(company_name),
            ipo_date=VALUES(ipo_date),
            years_listed=VALUES(years_listed),
            sector=VALUES(sector),
            industry=VALUES(industry),
            exchange=VALUES(exchange),
            market_cap=VALUES(market_cap),
            country=VALUES(country)
        """

        data = [
            (
                row['ticker'], row['company_name'], row['ipo_date'],
                row['years_listed'], row['sector'], row['industry'],
                row['exchange'], row['market_cap'], row['country']
            )
            for _, row in df.iterrows()
        ]

        cursor.executemany(insert_sql, data)
        conn.commit()

        print(f"✓ {len(data)}개 기업이 {table_name} 테이블에 저장되었습니다.")

        cursor.close()
        conn.close()

    except Exception as e:
        print(f"DB 저장 실패: {e}")


# ============================================================
# 실행 예시
# ============================================================
# if __name__ == "__main__":
#
#     # 1. 전체 주식 목록 가져오기
#     all_tickers = get_all_stock_list()
#
#     # 2. 테스트: 처음 100개만
#     # test_tickers = all_tickers[:100]
#
#     # 3. 필터링 실행
#     filtered_df = filter_mature_nonfinancial_companies(
#         tickers=all_tickers,
#         min_years=10,
#         exclude_sectors=['Financials', 'Utilities', 'Financial Services'],
#         batch_size=100,
#         save_interval=500
#     )
#
#     # 4. 결과 저장
#     if not filtered_df.empty:
#         filtered_df.to_csv('us_mature_nonfinancial_companies.csv', index=False)
#         print("✓ 결과가 CSV 파일로 저장되었습니다.")
#
#         # 통계 출력
#         show_sector_statistics(filtered_df)

        # DB 저장 (선택사항)
        # db_info = {
        #     'host': 'localhost',
        #     'user': 'your_user',
        #     'password': 'your_password',
        #     'database': 'your_database'
        # }
        # save_to_database(filtered_df, db_info)

In [4]:
# 1. 전체 주식 목록 가져오기
all_tickers = get_all_stock_list()

# 2. 테스트: 처음 100개만
# test_tickers = all_tickers[:100]

# 3. 필터링 실행
filtered_df = filter_mature_nonfinancial_companies(
    tickers=all_tickers,
    min_years=10,
    exclude_sectors=['Financials', 'Utilities', 'Financial Services'],
    batch_size=100,
    save_interval=500
)

총 12104개의 미국 상장 주식을 찾았습니다.

필터링 조건:
  - 최소 상장일: 2016-01-30 이전
  - 제외 섹터: Financials, Utilities, Financial Services
  - 총 티커 수: 12104



기업 정보 수집 중:   4%|▍         | 5/122 [08:10<3:11:35, 98.25s/it]


💾 중간 저장: 287개 기업 저장됨


기업 정보 수집 중:   8%|▊         | 10/122 [16:18<3:02:31, 97.78s/it]


💾 중간 저장: 521개 기업 저장됨


...


💾 중간 저장: 3559개 기업 저장됨


기업 정보 수집 중:  98%|█████████▊| 120/122 [3:14:06<03:11, 95.69s/it]


💾 중간 저장: 3676개 기업 저장됨


기업 정보 수집 중: 100%|██████████| 122/122 [3:15:43<00:00, 96.25s/it]


필터링 결과:
  - 조건 충족 기업: 3688개
  - 실패한 티커: 0개



In [6]:
# filtered_df.to_csv(r'C:\Users\82108\OneDrive\바탕 화면\investment\data\raw_data\screened_ticker_20260125.csv')

In [13]:
# 1. ticker 길이가 5자리가 아니고, '-' 기호가 없는 데이터만 필터링
filtered_tickers_df = filtered_df[
    (filtered_df['ticker'].str.len() != 5) &
    (~filtered_df['ticker'].str.contains('-', na=False))
].copy()

# 2. market_cap 기준 내림차순 정렬
filtered_tickers_df = filtered_tickers_df.sort_values('market_cap', ascending=False)

# 3. 상위 2000개 선택
top_2000_df = filtered_tickers_df.head(2000).reset_index(drop=True)

# 4. 순위 추가
top_2000_df['rank'] = range(1, len(top_2000_df) + 1)

# 5. ticker 리스트 추출
top_2000_tickers = top_2000_df['ticker'].tolist()

# 결과 확인
print(f"총 {len(top_2000_tickers)}개의 ticker 추출됨")
print(f"\n상위 20개:")
print(top_2000_df[['rank', 'ticker', 'sector', 'industry', 'market_cap']].head(20))

# 제외된 ticker 예시 확인
excluded_df = filtered_df[
    (filtered_df['ticker'].str.len() == 5) |
    (filtered_df['ticker'].str.contains('-', na=False))
]
print(f"\n제외된 ticker 수: {len(excluded_df)}")
print(f"제외된 ticker 예시: {excluded_df['ticker'].head(10).tolist()}")

# 2. 파일 생성
file_content = "ticker_list = ["

# 10개씩 한 줄에 배치
for i in range(0, len(top_2000_tickers), 10):
    batch = top_2000_tickers[i:i+10]
    if i == 0:
        line = "'" + "', '".join(batch) + "'"
    else:
        line = "    '" + "', '".join(batch) + "'"

    if i + 10 < len(top_2000_tickers):
        line += ","

    file_content += line + "\n"

file_content += "]"

# 3. 파일 저장
with open('../../../DATA/us_target_ticker_list_2000.py', 'w') as f:
    f.write(file_content)

print(f"총 {len(top_2000_tickers)}개의 ticker가 us_target_ticker_list_2000.py 파일로 저장되었습니다.")
print(f"\n처음 20개 ticker: {top_2000_tickers[:20]}")

총 2000개의 ticker 추출됨

상위 20개:
    rank ticker                  sector                        industry  \
0      1   NVDA              Technology                  Semiconductors   
1      2   GOOG              Technology  Internet Content & Information   
2      3   AAPL              Technology            Consumer Electronics   
3      4   MSFT              Technology       Software - Infrastructure   
4      5   AMZN       Consumer Cyclical                Specialty Retail   
5      6    TSM              Technology                  Semiconductors   
6      7   META              Technology  Internet Content & Information   
7      8   AVGO              Technology                  Semiconductors   
8      9   TSLA       Consumer Cyclical            Auto - Manufacturers   
9     10    LLY              Healthcare    Drug Manufacturers - General   
10    11    WMT      Consumer Defensive                 Discount Stores   
11    12    XOM                  Energy            Oil & Gas Integrated